In [8]:
!pip install -q comet_ml


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
import math
import time
import random
from dataclasses import dataclass
from collections import deque, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [9]:
os.environ["COMET_API_KEY"] = "E0COdo2ExU3AtjEVV0Rh1i393" 
os.environ["COMET_MODE"] = "online"

In [10]:
USE_COMET = True

if USE_COMET:
    from comet_ml import Experiment

def create_experiment(run_name: str, params: dict):
    if not USE_COMET:
        return None

    experiment = Experiment(
        api_key=os.environ["COMET_API_KEY"],
        project_name="battle-city-rl",
        workspace=os.environ.get("COMET_WORKSPACE"),
    )
    experiment.set_name(run_name)
    experiment.log_parameters(params)
    return experiment

In [27]:
from dataclasses import dataclass

@dataclass(frozen=True)
class StateIndexer:
    n_cells: int          # например, 9x9 -> 81
    n_dirs: int = 4
    n_bullet: int = 2

    @property
    def n_states(self) -> int:
        return self.n_cells * self.n_dirs * self.n_cells * self.n_dirs * self.n_bullet

    def encode(self, s_t) -> int:
        """
        s_t = (p_idx, p_dir, e_idx, e_dir, bullet_alive)
        """
        p_idx, p_dir, e_idx, e_dir, bullet_alive = s_t

        idx = int(p_idx)
        idx = idx * self.n_dirs + int(p_dir)
        idx = idx * self.n_cells + int(e_idx)
        idx = idx * self.n_dirs + int(e_dir)
        idx = idx * self.n_bullet + int(bullet_alive)
        return idx

    def decode(self, idx: int):
        bullet_alive = idx % self.n_bullet
        idx //= self.n_bullet

        e_dir = idx % self.n_dirs
        idx //= self.n_dirs

        e_idx = idx % self.n_cells
        idx //= self.n_cells

        p_dir = idx % self.n_dirs
        idx //= self.n_dirs

        p_idx = idx
        return (p_idx, p_dir, e_idx, e_dir, bullet_alive)

COMET ERROR: Due to connectivity issues, there's an error in processing the heartbeat. The experiment's status updates might be inaccurate until the connection issues are resolved.


In [11]:
# ПРИМЕР. ЗАМЕНИ НА РЕАЛЬНЫЙ ИМПОРТ ИЗ tanks-rl
# from your_repo.env import BattleCityEnv

# Заглушка: сюда подставь реальный конструктор среды
def make_base_env():
    """
    Верни здесь экземпляр среды из репозитория.
    Например:
        return BattleCityEnv(grid_size=9)
    """
    raise NotImplementedError("Подставь реальный импорт среды из tanks-rl")

In [12]:
# ЗАМЕНИ при необходимости под action mapping из твоего env
NOOP = 0
MOVE_UP = 1
MOVE_RIGHT = 2
MOVE_DOWN = 3
MOVE_LEFT = 4
SHOOT = 5

UP = 0
RIGHT = 1
DOWN = 2
LEFT = 3

N_ACTIONS = 6

In [13]:
def to_discrete_state(obs, env=None):
    """
    Должна вернуть дискретное состояние в форме:
        s_t = (p_idx, p_dir, e_idx, e_dir, bullet_alive)

    По умолчанию предполагается, что obs уже такой tuple.
    """
    return obs

In [15]:
class IndexedEnv:
    def __init__(self, env, state_indexer: StateIndexer):
        self.env = env
        self.state_indexer = state_indexer

        # Если у среды есть action_space.n, используем его
        self.n_actions = getattr(getattr(env, "action_space", None), "n", N_ACTIONS)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        s_t = to_discrete_state(obs, self.env)
        return self.state_indexer.encode(s_t), info

    def step(self, a_t: int):
        obs, r_t, terminated, truncated, info = self.env.step(a_t)
        s_tp1 = to_discrete_state(obs, self.env)
        return self.state_indexer.encode(s_tp1), r_t, terminated, truncated, info

    def __getattr__(self, name):
        return getattr(self.env, name)

In [16]:
class TabularQAgent:
    def __init__(
        self,
        n_states: int,
        n_actions: int,
        gamma: float = 0.99,
        alpha_crit: float = 0.1,
        epsilon: float = 1.0,
        epsilon_min: float = 0.05,
        epsilon_decay: float = 0.9995,
        seed: int = 42,
    ):
        self.n_states = n_states
        self.n_actions = n_actions
        self.gamma = gamma
        self.alpha_crit = alpha_crit
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.rng = np.random.default_rng(seed)

        self.q = np.zeros((n_states, n_actions), dtype=np.float32)

    def pi(self, s_t: int, greedy: bool = False) -> int:
        if (not greedy) and (self.rng.random() < self.epsilon):
            return int(self.rng.integers(self.n_actions))
        return int(np.argmax(self.q[s_t]))

    def update(self, s_t: int, a_t: int, r_t: float, s_tp1: int, done: bool):
        if done:
            target = r_t
        else:
            target = r_t + self.gamma * float(np.max(self.q[s_tp1]))

        self.q[s_t, a_t] = (
            (1.0 - self.alpha_crit) * self.q[s_t, a_t]
            + self.alpha_crit * target
        )

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [17]:
class DoubleQAgent:
    def __init__(
        self,
        n_states: int,
        n_actions: int,
        gamma: float = 0.99,
        alpha_crit: float = 0.1,
        epsilon: float = 1.0,
        epsilon_min: float = 0.05,
        epsilon_decay: float = 0.9995,
        seed: int = 42,
    ):
        self.n_states = n_states
        self.n_actions = n_actions
        self.gamma = gamma
        self.alpha_crit = alpha_crit
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.rng = np.random.default_rng(seed)

        self.q_A = np.zeros((n_states, n_actions), dtype=np.float32)
        self.q_B = np.zeros((n_states, n_actions), dtype=np.float32)

    def pi(self, s_t: int, greedy: bool = False) -> int:
        if (not greedy) and (self.rng.random() < self.epsilon):
            return int(self.rng.integers(self.n_actions))
        return int(np.argmax(self.q_A[s_t] + self.q_B[s_t]))

    def update(self, s_t: int, a_t: int, r_t: float, s_tp1: int, done: bool):
        if self.rng.random() < 0.5:
            if done:
                target = r_t
            else:
                a_star = int(np.argmax(self.q_A[s_tp1]))
                target = r_t + self.gamma * float(self.q_B[s_tp1, a_star])

            self.q_A[s_t, a_t] = (
                (1.0 - self.alpha_crit) * self.q_A[s_t, a_t]
                + self.alpha_crit * target
            )
        else:
            if done:
                target = r_t
            else:
                a_star = int(np.argmax(self.q_B[s_tp1]))
                target = r_t + self.gamma * float(self.q_A[s_tp1, a_star])

            self.q_B[s_t, a_t] = (
                (1.0 - self.alpha_crit) * self.q_B[s_t, a_t]
                + self.alpha_crit * target
            )

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [18]:
class ExpectedSARSAAgent:
    def __init__(
        self,
        n_states: int,
        n_actions: int,
        gamma: float = 0.99,
        alpha_crit: float = 0.1,
        epsilon: float = 1.0,
        epsilon_min: float = 0.05,
        epsilon_decay: float = 0.9995,
        seed: int = 42,
    ):
        self.n_states = n_states
        self.n_actions = n_actions
        self.gamma = gamma
        self.alpha_crit = alpha_crit
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.rng = np.random.default_rng(seed)

        self.q = np.zeros((n_states, n_actions), dtype=np.float32)

    def pi(self, s_t: int, greedy: bool = False) -> int:
        if (not greedy) and (self.rng.random() < self.epsilon):
            return int(self.rng.integers(self.n_actions))
        return int(np.argmax(self.q[s_t]))

    def policy_probs(self, s_t: int):
        probs = np.full(self.n_actions, self.epsilon / self.n_actions, dtype=np.float32)
        a_star = int(np.argmax(self.q[s_t]))
        probs[a_star] += 1.0 - self.epsilon
        return probs

    def update(self, s_t: int, a_t: int, r_t: float, s_tp1: int, done: bool):
        if done:
            target = r_t
        else:
            probs = self.policy_probs(s_tp1)
            expected_q = float(np.dot(probs, self.q[s_tp1]))
            target = r_t + self.gamma * expected_q

        self.q[s_t, a_t] = (
            (1.0 - self.alpha_crit) * self.q[s_t, a_t]
            + self.alpha_crit * target
        )

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

In [19]:
def aligned(x, y):
    return x[0] == y[0] or x[1] == y[1]

def facing_target(player_pos, enemy_pos, player_dir):
    if player_pos[0] == enemy_pos[0]:
        return (enemy_pos[1] > player_pos[1] and player_dir == RIGHT) or \
               (enemy_pos[1] < player_pos[1] and player_dir == LEFT)
    if player_pos[1] == enemy_pos[1]:
        return (enemy_pos[0] > player_pos[0] and player_dir == DOWN) or \
               (enemy_pos[0] < player_pos[0] and player_dir == UP)
    return False

def move_towards(src, dst):
    dr = dst[0] - src[0]
    dc = dst[1] - src[1]
    if abs(dr) >= abs(dc):
        return MOVE_DOWN if dr > 0 else MOVE_UP
    return MOVE_RIGHT if dc > 0 else MOVE_LEFT

def manhattan(x, y):
    return abs(x[0] - y[0]) + abs(x[1] - y[1])

class LineOfSightPolicy:
    def __init__(self, blocking_cells):
        self.blocking_cells = set(blocking_cells)

    def clear_line(self, grid, src, dst):
        r1, c1 = src
        r2, c2 = dst

        if r1 == r2:
            step = 1 if c2 > c1 else -1
            for c in range(c1 + step, c2, step):
                if grid[r1][c] in self.blocking_cells:
                    return False
            return True

        if c1 == c2:
            step = 1 if r2 > r1 else -1
            for r in range(r1 + step, r2, step):
                if grid[r][c1] in self.blocking_cells:
                    return False
            return True

        return False

    def pi(self, env) -> int:
        player_pos = tuple(env.player_pos)
        enemy_pos = tuple(env.enemy_pos)
        player_dir = int(env.player_dir)
        grid = env.grid

        if aligned(player_pos, enemy_pos) and self.clear_line(grid, player_pos, enemy_pos):
            if facing_target(player_pos, enemy_pos, player_dir):
                return SHOOT
            return move_towards(player_pos, enemy_pos)

        return move_towards(player_pos, enemy_pos)

class CastleDefenderPolicy:
    def __init__(self, blocking_cells, danger_radius: int = 3):
        self.danger_radius = danger_radius
        self.los = LineOfSightPolicy(blocking_cells)

    def pi(self, env) -> int:
        player_pos = tuple(env.player_pos)
        enemy_pos = tuple(env.enemy_pos)
        castle_pos = tuple(env.castle_pos)

        if manhattan(enemy_pos, castle_pos) <= self.danger_radius:
            return move_towards(player_pos, castle_pos)

        return self.los.pi(env)

In [20]:
class MetricTracker:
    def __init__(self, window: int = 100):
        self.window = window
        self.episode_returns = deque(maxlen=window)
        self.episode_scores = deque(maxlen=window)
        self.win_rate = deque(maxlen=window)
        self.castle_survival = deque(maxlen=window)
        self.episode_length = deque(maxlen=window)

        self.history = defaultdict(list)

    def update(self, episode_return, episode_score, win, castle_alive, ep_len):
        self.episode_returns.append(float(episode_return))
        self.episode_scores.append(float(episode_score))
        self.win_rate.append(float(win))
        self.castle_survival.append(float(castle_alive))
        self.episode_length.append(int(ep_len))

        self.history["episode_return"].append(float(episode_return))
        self.history["episode_score"].append(float(episode_score))
        self.history["win"].append(float(win))
        self.history["castle_alive"].append(float(castle_alive))
        self.history["ep_len"].append(int(ep_len))

    def rolling_summary(self):
        if len(self.episode_returns) == 0:
            return {}

        return {
            "Average Return": float(np.mean(self.episode_returns)),
            "Average Score": float(np.mean(self.episode_scores)),
            "Max Score": float(np.max(self.episode_scores)),
            "Win Rate": float(np.mean(self.win_rate)),
            "Castle Survival": float(np.mean(self.castle_survival)),
            "Average Episode Length": float(np.mean(self.episode_length)),
        }

In [21]:
def evaluate_agent(make_env_fn, state_indexer, agent, n_eval_episodes=50):
    scores = []
    returns = []
    wins = []
    castle_survivals = []
    lengths = []

    for _ in range(n_eval_episodes):
        env_raw = make_env_fn()
        env = IndexedEnv(env_raw, state_indexer)

        s_t, info = env.reset()
        done = False

        episode_return = 0.0
        episode_score = 0.0
        ep_len = 0

        while not done:
            a_t = agent.pi(s_t, greedy=True)
            s_tp1, r_t, terminated, truncated, info = env.step(a_t)
            done = terminated or truncated

            episode_return += r_t
            episode_score = info.get("score", episode_return)
            ep_len += 1
            s_t = s_tp1

        scores.append(float(episode_score))
        returns.append(float(episode_return))
        wins.append(float(info.get("win", False)))
        castle_survivals.append(float(info.get("castle_survived", True)))
        lengths.append(ep_len)

        if hasattr(env_raw, "close"):
            env_raw.close()

    return {
        "Max Score": float(np.max(scores)),
        "Average Score": float(np.mean(scores)),
        "Average Return": float(np.mean(returns)),
        "Win Rate": float(np.mean(wins)),
        "Castle Survival": float(np.mean(castle_survivals)),
        "Average Episode Length": float(np.mean(lengths)),
    }


def evaluate_heuristic(make_env_fn, policy, n_eval_episodes=50):
    scores = []
    returns = []
    wins = []
    castle_survivals = []
    lengths = []

    for _ in range(n_eval_episodes):
        env = make_env_fn()
        obs, info = env.reset()
        done = False

        episode_return = 0.0
        episode_score = 0.0
        ep_len = 0

        while not done:
            a_t = policy.pi(env)
            obs, r_t, terminated, truncated, info = env.step(a_t)
            done = terminated or truncated

            episode_return += r_t
            episode_score = info.get("score", episode_return)
            ep_len += 1

        scores.append(float(episode_score))
        returns.append(float(episode_return))
        wins.append(float(info.get("win", False)))
        castle_survivals.append(float(info.get("castle_survived", True)))
        lengths.append(ep_len)

        if hasattr(env, "close"):
            env.close()

    return {
        "Max Score": float(np.max(scores)),
        "Average Score": float(np.mean(scores)),
        "Average Return": float(np.mean(returns)),
        "Win Rate": float(np.mean(wins)),
        "Castle Survival": float(np.mean(castle_survivals)),
        "Average Episode Length": float(np.mean(lengths)),
    }

In [22]:
def train_agent(
    make_env_fn,
    state_indexer: StateIndexer,
    agent,
    n_episodes: int = 2000,
    eval_every: int = 200,
    n_eval_episodes: int = 50,
    run_name: str = "run",
    experiment=None,
):
    tracker = MetricTracker(window=100)
    eval_history = []

    start_time = time.time()

    for i in range(1, n_episodes + 1):
        env_raw = make_env_fn()
        env = IndexedEnv(env_raw, state_indexer)

        s_t, info = env.reset()
        done = False

        episode_return = 0.0
        episode_score = 0.0
        ep_len = 0

        while not done:
            a_t = agent.pi(s_t, greedy=False)
            s_tp1, r_t, terminated, truncated, info = env.step(a_t)
            done = terminated or truncated

            agent.update(s_t, a_t, r_t, s_tp1, done)

            episode_return += r_t
            episode_score = info.get("score", episode_return)
            ep_len += 1
            s_t = s_tp1

        agent.decay_epsilon()

        tracker.update(
            episode_return=episode_return,
            episode_score=episode_score,
            win=info.get("win", False),
            castle_alive=info.get("castle_survived", True),
            ep_len=ep_len,
        )

        if experiment is not None:
            experiment.log_metric("Train/Episode Return", episode_return, step=i)
            experiment.log_metric("Train/Episode Score", episode_score, step=i)
            experiment.log_metric("Train/Epsilon", agent.epsilon, step=i)

            rolling = tracker.rolling_summary()
            for k, v in rolling.items():
                experiment.log_metric(f"Train/{k}", v, step=i)

        if i % eval_every == 0:
            eval_metrics = evaluate_agent(
                make_env_fn=make_env_fn,
                state_indexer=state_indexer,
                agent=agent,
                n_eval_episodes=n_eval_episodes,
            )
            eval_metrics["iteration"] = i
            eval_history.append(eval_metrics)

            if experiment is not None:
                experiment.log_metrics(
                    {f"Eval/{k}": v for k, v in eval_metrics.items() if k != "iteration"},
                    step=i,
                )

            print(f"[{run_name}] iteration={i} | {eval_metrics}")

        if hasattr(env_raw, "close"):
            env_raw.close()

    total_time = time.time() - start_time

    return {
        "agent": agent,
        "tracker": tracker,
        "eval_history": pd.DataFrame(eval_history),
        "train_time_sec": total_time,
    }

In [23]:
def moving_average(x, w=50):
    x = np.asarray(x, dtype=np.float32)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

def plot_training_curves(results_dict):
    plt.figure(figsize=(12, 5))

    for run_name, result in results_dict.items():
        y = result["tracker"].history["episode_return"]
        y_smooth = moving_average(y, w=50)
        plt.plot(y_smooth, label=run_name)

    plt.title("Training: Episode Return (moving average)")
    plt.xlabel("Episode")
    plt.ylabel("Return")
    plt.legend()
    plt.grid(True)
    plt.show()


def plot_eval_metric(results_dict, metric_name="Average Score"):
    plt.figure(figsize=(12, 5))

    for run_name, result in results_dict.items():
        df = result["eval_history"]
        if len(df) == 0:
            continue
        plt.plot(df["iteration"], df[metric_name], marker="o", label=run_name)

    plt.title(f"Evaluation: {metric_name}")
    plt.xlabel("Iteration")
    plt.ylabel(metric_name)
    plt.legend()
    plt.grid(True)
    plt.show()

In [24]:
GRID_H = 9
GRID_W = 9
N_CELLS = GRID_H * GRID_W

state_indexer = StateIndexer(
    n_cells=N_CELLS,
    n_dirs=4,
    n_bullet=2,
)

CONFIG = {
    "gamma": 0.99,
    "alpha_crit": 0.1,
    "epsilon": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.999,
    "n_episodes": 2000,
    "eval_every": 200,
    "n_eval_episodes": 50,
}

state_indexer.n_states

209952

In [25]:
def make_agents():
    return {
        "q_learning": TabularQAgent(
            n_states=state_indexer.n_states,
            n_actions=N_ACTIONS,
            gamma=CONFIG["gamma"],
            alpha_crit=CONFIG["alpha_crit"],
            epsilon=CONFIG["epsilon"],
            epsilon_min=CONFIG["epsilon_min"],
            epsilon_decay=CONFIG["epsilon_decay"],
            seed=SEED,
        ),
        "double_q": DoubleQAgent(
            n_states=state_indexer.n_states,
            n_actions=N_ACTIONS,
            gamma=CONFIG["gamma"],
            alpha_crit=CONFIG["alpha_crit"],
            epsilon=CONFIG["epsilon"],
            epsilon_min=CONFIG["epsilon_min"],
            epsilon_decay=CONFIG["epsilon_decay"],
            seed=SEED,
        ),
        "expected_sarsa": ExpectedSARSAAgent(
            n_states=state_indexer.n_states,
            n_actions=N_ACTIONS,
            gamma=CONFIG["gamma"],
            alpha_crit=CONFIG["alpha_crit"],
            epsilon=CONFIG["epsilon"],
            epsilon_min=CONFIG["epsilon_min"],
            epsilon_decay=CONFIG["epsilon_decay"],
            seed=SEED,
        ),
    }

In [26]:
agents = make_agents()
results = {}

for run_name, agent in agents.items():
    experiment = create_experiment(run_name, {**CONFIG, "agent": run_name})

    result = train_agent(
        make_env_fn=make_base_env,
        state_indexer=state_indexer,
        agent=agent,
        n_episodes=CONFIG["n_episodes"],
        eval_every=CONFIG["eval_every"],
        n_eval_episodes=CONFIG["n_eval_episodes"],
        run_name=run_name,
        experiment=experiment,
    )

    results[run_name] = result

    if experiment is not None:
        experiment.end()

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/yomanfeed/battle-city-rl/a36bbe8b66dc45029eaec4a52bdc24ec



NotImplementedError: Подставь реальный импорт среды из tanks-rl